In [ ]:
import os
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except Exception:
    torch = None

FILE_NAMES = ["bn_identity_in_ratio_radar_final_ver0.pkl"]

all_data = defaultdict(list)
for file_name in FILE_NAMES:
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"Missing file: {file_name} (cwd: {os.getcwd()})")
    with open(file_name, "rb") as file:
        loaded_data = pickle.load(file)
    for key, value in loaded_data.items():
        all_data[key].extend(value)

all_data = dict(all_data)
print(f"total values: {sum(len(v) for v in all_data.values())}")

METRIC_GROUPS = {
    "Correlation": [
        "rad_perturb_rad_correlation",
        "rad_perturb_cam_correlation",
        "cam_perturb_rad_correlation",
        "cam_perturb_cam_correlation",
    ],
    "Cosine Similarity": [
        "rad_perturb_rad_cosine_similarity",
        "rad_perturb_cam_cosine_similarity",
        "cam_perturb_rad_cosine_similarity",
        "cam_perturb_cam_cosine_similarity",
    ],
    "MSE": [
        "rad_perturb_rad_MSE",
        "rad_perturb_cam_MSE",
        "cam_perturb_rad_MSE",
        "cam_perturb_cam_MSE",
    ],
    "IOU": [
        "rad_perturb_rad_IOU",
        "rad_perturb_cam_IOU",
        "cam_perturb_rad_IOU",
        "cam_perturb_cam_IOU",
    ],
}
HIST_KEY = "rad_perturb_cam_l2_norm"

required_keys = {HIST_KEY}
for keys in METRIC_GROUPS.values():
    required_keys.update(keys)

missing = sorted(k for k in required_keys if k not in all_data)
if missing:
    raise KeyError(f"Missing keys in pickle: {missing}")

def to_float(value):
    if torch is not None and torch.is_tensor(value):
        return value.detach().cpu().item()
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return float(value)

data_cpu = {k: [to_float(v) for v in all_data[k]] for k in required_keys}
data_df = pd.DataFrame(data_cpu)

WINDOW_SIZE = 100
smoothed = data_df.rolling(window=WINDOW_SIZE, min_periods=1).mean()
means = smoothed.mean()
stds = smoothed.std()


In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
})

def plot_group(title, keys):
    ax = smoothed[keys].plot(
        figsize=(10, 5),
        title=title,
        xlabel="Index",
        ylabel="Value",
        grid=True,
    )
    labels = [f"{k}: mean={means[k]:.2f} +/- {stds[k]:.4f}" for k in keys]
    ax.legend(labels, title=title)
    plt.show()

for title, keys in METRIC_GROUPS.items():
    plot_group(f"Smoothed {title}", keys)

values = data_df[HIST_KEY].dropna().to_numpy()
if values.size == 0:
    raise ValueError(f"No values found for {HIST_KEY}")

mean_value = np.mean(values)
median_value = np.median(values)
std_dev = np.std(values)

plt.figure(figsize=(10, 6))
plt.hist(values, bins=100, density=True, alpha=0.6, color="b", label="Histogram")

try:
    import scipy.stats as stats
    density = stats.gaussian_kde(values)
    x = np.linspace(values.min(), values.max(), 1000)
    plt.plot(x, density(x), label="KDE", color="red")
except Exception:
    print("scipy not available; skipping KDE")

plt.axvline(mean_value, color="black", linestyle="--", linewidth=1.5, label=f"Mean = {mean_value:.2f}")
plt.axvline(mean_value - std_dev, color="purple", linestyle=":", linewidth=1.2, label=f"1 SD = {std_dev:.4f}")
plt.axvline(mean_value + std_dev, color="purple", linestyle=":", linewidth=1.2)
plt.axvline(mean_value - 2 * std_dev, color="green", linestyle=":", linewidth=1.2, label=f"Median = {median_value:.2f}")
plt.axvline(mean_value + 2 * std_dev, color="green", linestyle=":", linewidth=1.2)

plt.title("Histogram and KDE of L2 Norm")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.show()
